### Librerías a utilizar
---

In [40]:
import pickle
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import  root_mean_squared_error
from sklearn.feature_extraction import  DictVectorizer
from sklearn.preprocessing import StandardScaler
from dotenv import load_dotenv
import math
import optuna
import pathlib
from optuna.samplers import TPESampler
from mlflow.models.signature import infer_signature
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from mlflow import MlflowClient
from datetime import datetime
import mlflow.pyfunc as mlflow_pyfunc
from statsmodels.stats.outliers_influence import variance_inflation_factor
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split


### Cargar las credenciales
---

In [41]:
load_dotenv(override=True)  # Carga las variables del archivo .env
EXPERIMENT_NAME = "/Users/sarahbeltrang@gmail.com/project1-experiment" 

mlflow.set_tracking_uri("databricks")
experiment = mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

### Preprocessing
___

In [ ]:
#def preprocessing(df: pd.DataFrame):
    # Quitar columnas
    #df = df.drop(columns=['Health_Issues', 'Caffeine_mg'], errors='ignore')

    # Filtrar filas con género "Other"
    #df = df[df["Gender"] != "Other"]

    # Mapear países a continentes 
    #pais_a_continente = {
        #"Canada": "America", "USA": "America", "Mexico": "America", "Brazil": "America",
        #"Norway": "Europe", "Sweden": "Europe", "UK": "Europe", "Finland": "Europe",
        #"Italy": "Europe", "Belgium": "Europe", "Germany": "Europe", "France": "Europe",
        #"Switzerland": "Europe", "Netherlands": "Europe", "Spain": "Europe",
        #"India": "Asia", "China": "Asia", "South Korea": "Asia", "Japan": "Asia",
        #"Australia": "Oceania"
    #}
    #df["Continent"] = df["Country"].map(pais_a_continente)

    # Mapear variables categóricas 
    #df['Sleep_Quality'] = df['Sleep_Quality'].map({'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3})
    #df['Stress_Level'] = df['Stress_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
    #df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})

    # Eliminar columnas
    #df = df.drop(columns=["Country"], errors='ignore')
    #if "ID" in df.columns:
        #df = df.drop(columns=["ID"])

    # Convertir booleanos a enteros 
    #for col in df.columns:
        #if df[col].dtype == 'bool':
            #df[col] = df[col].astype(int)

    # Convertir a matriz con DictVectorizer 
    #dicts = df.drop(columns=["Stress_Level"]).to_dict(orient="records")
    #dv = DictVectorizer(sparse=False)
    #X = dv.fit_transform(dicts)

    # Target 
    #y = df["Stress_Level"].values

    # Balancear con SMOTE 
    #smote = SMOTE(random_state=42)
    #X_bal, y_bal = smote.fit_resample(X, y)

    #print("Preprocessing completado.")
    #print("Shape features balanceadas:", X_bal.shape)
    #print("Distribución target balanceado:\n", pd.Series(y_bal).value_counts())

    #return X_bal, y_bal, dv

In [42]:
def preprocessing_train(df: pd.DataFrame, vif_threshold=5):
    # Mantener la misma firma mínima que pediste (solo lo necesario)
    df = df.drop(columns=['Health_Issues', 'Caffeine_mg', 'Sleep_Hours', 'Sleep_Quality'], errors='ignore')
    df = df[df["Gender"] != "Other"]

    pais_a_continente = {
        "Canada": "America", "USA": "America", "Mexico": "America", "Brazil": "America",
        "Norway": "Europe", "Sweden": "Europe", "UK": "Europe", "Finland": "Europe",
        "Italy": "Europe", "Belgium": "Europe", "Germany": "Europe", "France": "Europe",
        "Switzerland": "Europe", "Netherlands": "Europe", "Spain": "Europe",
        "India": "Asia", "China": "Asia", "South Korea": "Asia", "Japan": "Asia",
        "Australia": "Oceania"
    }
    df["Continent"] = df["Country"].map(pais_a_continente)

    df['Stress_Level'] = df['Stress_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
    df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})

    df = df.drop(columns=["Country"], errors='ignore')
    if "ID" in df.columns:
        df = df.drop(columns=["ID"])

    for col in df.columns:
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)

    # Separar X e y
    y = df["Stress_Level"].values
    X_df = df.drop(columns=["Stress_Level"])

    # DictVectorizer
    dicts = X_df.to_dict(orient="records")
    dv = DictVectorizer(sparse=False)
    X = dv.fit_transform(dicts)
    X_df_encoded = pd.DataFrame(X, columns=dv.get_feature_names_out())

    # --- Limpieza mínima antes de VIF ---
    # 1) reemplazar inf por NaN y luego rellenar NaN con 0
    X_df_encoded = X_df_encoded.replace([np.inf, -np.inf], np.nan)
    if X_df_encoded.isna().any().any():
        X_df_encoded = X_df_encoded.fillna(0)

    # 2) eliminar una dummy por cada categoría para evitar colinealidad perfecta
    feature_names = dv.get_feature_names_out().tolist()
    groups = {}
    for f in feature_names:
        if '=' in f:
            pref = f.split('=')[0]
            groups.setdefault(pref, []).append(f)
    to_drop = []
    for pref, feats in groups.items():
        if len(feats) > 1:
            feats_sorted = sorted(feats)
            to_drop.append(feats_sorted[0])
    if to_drop:
        X_df_encoded = X_df_encoded.drop(columns=[c for c in to_drop if c in X_df_encoded.columns], errors='ignore')

    # 3) eliminar columnas constantes
    nunique = X_df_encoded.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()
    if constant_cols:
        X_df_encoded = X_df_encoded.drop(columns=constant_cols, errors='ignore')

    # --- Filtrado VIF (defensivo) ---
    features = list(X_df_encoded.columns)
    dropped = []
    while True:
        if len(features) < 2:
            break
        vif_vals = []
        for i in range(len(features)):
            try:
                v = variance_inflation_factor(X_df_encoded[features].values, i)
                if not np.isfinite(v):
                    v = np.inf
            except Exception:
                v = np.inf
            vif_vals.append(v)

        vif_data = pd.DataFrame({"variable": features, "VIF": vif_vals})

        # eliminar VIF infinito primero
        if np.isinf(vif_data["VIF"]).any():
            variable_to_drop = vif_data.loc[vif_data["VIF"].idxmax(), "variable"]
            dropped.append((variable_to_drop, float('inf')))
            features.remove(variable_to_drop)
            print(f"VIF infinito -> Eliminando: {variable_to_drop}")
            continue

        max_vif = vif_data["VIF"].max()
        if max_vif > vif_threshold:
            variable_to_drop = vif_data.loc[vif_data["VIF"].idxmax(), "variable"]
            dropped.append((variable_to_drop, float(max_vif)))
            features.remove(variable_to_drop)
            print(f"Eliminando por VIF alto: {variable_to_drop} ({max_vif:.2f})")
        else:
            break

    # DataFrame con features finales (antes de scale)
    X_filtered_df = X_df_encoded[features]

    # --- Escalado (fit en train) ---
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_filtered_df.values)

    # --- SMOTE (usar matriz escalada) ---
    smote = SMOTE(random_state=42)
    X_bal, y_bal = smote.fit_resample(X_scaled, y)

    print("Preprocessing train completado.")
    print("Shape features balanceadas (train):", X_bal.shape)
    print("Distribución target balanceado (train):\n", pd.Series(y_bal).value_counts())

    # devolver scaler para usar en evaluación
    return X_bal, y_bal, dv, features, dropped, scaler

In [43]:
def preprocessing_eval(df: pd.DataFrame, dv: DictVectorizer, features, scaler: StandardScaler):
    # Versión mínima compatible + aplicación de scaler
    df = df.drop(columns=['Health_Issues', 'Caffeine_mg', 'Sleep_Hours', 'Sleep_Quality'], errors='ignore')
    df = df[df["Gender"] != "Other"]

    pais_a_continente = {
        "Canada": "America", "USA": "America", "Mexico": "America", "Brazil": "America",
        "Norway": "Europe", "Sweden": "Europe", "UK": "Europe", "Finland": "Europe",
        "Italy": "Europe", "Belgium": "Europe", "Germany": "Europe", "France": "Europe",
        "Switzerland": "Europe", "Netherlands": "Europe", "Spain": "Europe",
        "India": "Asia", "China": "Asia", "South Korea": "Asia", "Japan": "Asia",
        "Australia": "Oceania"
    }
    df["Continent"] = df["Country"].map(pais_a_continente)

    df['Stress_Level'] = df['Stress_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
    df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
    df = df.drop(columns=["Country"], errors='ignore')
    if "ID" in df.columns:
        df = df.drop(columns=["ID"])

    for col in df.columns:
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)

    dicts = df.drop(columns=["Stress_Level"]).to_dict(orient="records")
    X_encoded = dv.transform(dicts).astype(float)

    X_encoded[~np.isfinite(X_encoded)] = np.nan
    if np.isnan(X_encoded).any():
        X_encoded = np.nan_to_num(X_encoded, nan=0.0, posinf=0.0, neginf=0.0)

    feature_names = dv.get_feature_names_out()
    X_df_encoded = pd.DataFrame(X_encoded, columns=feature_names)

    # eliminar mismas dummies base (primera dummy por prefijo)
    groups = {}
    for f in feature_names:
        if '=' in f:
            pref = f.split('=')[0]
            groups.setdefault(pref, []).append(f)
    to_drop = []
    for pref, feats in groups.items():
        if len(feats) > 1:
            feats_sorted = sorted(feats)
            to_drop.append(feats_sorted[0])
    if to_drop:
        X_df_encoded = X_df_encoded.drop(columns=[c for c in to_drop if c in X_df_encoded.columns], errors='ignore')

    # Asegurar que todas las features estén presentes (si falta alguna, añadir con 0)
    for f in features:
        if f not in X_df_encoded.columns:
            X_df_encoded[f] = 0.0

    # Seleccionar columnas en mismo orden que 'features'
    X_filtered_df = X_df_encoded[features]

    # Transformar con scaler (fue ajustado en train)
    X_scaled = scaler.transform(X_filtered_df.values)

    y = df["Stress_Level"].values
    return X_scaled, y

In [ ]:
#target = 'Stress_Level'  
#X = df.drop(columns=[target])
#y = df[target].values

### Dividir en entrenamiento, prueba & validacion
---

In [44]:
df_raw = pd.read_csv("../data/raw/synthetic_coffee_health_10000.csv")

In [45]:
target = 'Stress_Level'

In [46]:
train_df, temp_df = train_test_split(df_raw, test_size=0.4, random_state=42, stratify=df_raw[target])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df[target])

In [47]:
# Preprocess train (fit dv + SMOTE)
X_train_bal, y_train_bal, dv, features, dropped, scaler = preprocessing_train(train_df, vif_threshold=7)

Eliminando por VIF alto: Heart_Rate (29.62)
Eliminando por VIF alto: BMI (16.34)


2025/11/10 17:36:57 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '1d546844428a4813a0187180685cfda9', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2025/11/10 17:36:59 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels were not specified. To automatically record training metrics, provide training labels as inputs to the model training function.


🏃 View run glamorous-fish-644 at: https://dbc-905d578f-b3ee.cloud.databricks.com/ml/experiments/2774895092678305/runs/1d546844428a4813a0187180685cfda9
🧪 View experiment at: https://dbc-905d578f-b3ee.cloud.databricks.com/ml/experiments/2774895092678305


2025/11/10 17:36:59 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '4e07377cac504a60a85ed4b16f0a5ff0', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2025/11/10 17:37:00 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels were not specified. To automatically record training metrics, provide training labels as inputs to the model training function.


🏃 View run smiling-swan-448 at: https://dbc-905d578f-b3ee.cloud.databricks.com/ml/experiments/2774895092678305/runs/4e07377cac504a60a85ed4b16f0a5ff0
🧪 View experiment at: https://dbc-905d578f-b3ee.cloud.databricks.com/ml/experiments/2774895092678305
Preprocessing train completado.
Shape features balanceadas (train): (12285, 13)
Distribución target balanceado (train):
 0    4095
2    4095
1    4095
Name: count, dtype: int64


In [48]:
print("Features seleccionadas (después de VIF):")
print(features)

Features seleccionadas (después de VIF):
['Age', 'Alcohol_Consumption', 'Coffee_Intake', 'Continent=Asia', 'Continent=Europe', 'Continent=Oceania', 'Gender', 'Occupation=Office', 'Occupation=Other', 'Occupation=Service', 'Occupation=Student', 'Physical_Activity_Hours', 'Smoking']


In [49]:
X_val, y_val = preprocessing_eval(val_df, dv, features, scaler)
X_test, y_test = preprocessing_eval(test_df, dv, features, scaler)

In [ ]:
#X_train, X_test_val, y_train, y_test_val = train_test_split(X, y, test_size=0.4, random_state=42)

#X_val, X_test, y_val, y_test = train_test_split(X_test_val, y_test_val, test_size=0.5, random_state=42)

### Regresión Logística
---

#### Función objetivo

In [ ]:
# def objective_logreg(trial: optuna.trial.Trial):
#     # Hiperparámetros a buscar
#     penalty = trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"])
#     l1_ratio = None
#     if penalty == "elasticnet":
#         l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)

#     # Construir param dict para logging y modelo
#     params = {
#         "penalty": penalty,
#         "C": trial.suggest_float("C", math.exp(-7), 1e2, log=True),
#         "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
#         "solver": "saga",           # saga soporta l1, elasticnet y multinomial
#         "multi_class": "multinomial",
#         "max_iter": 500,
#         "random_state": 42,
#         "n_jobs": -1
#     }
#     if penalty == "elasticnet":
#         params["l1_ratio"] = l1_ratio
#         params["penalty"] = "elasticnet"

#     with mlflow.start_run(nested=True):
#         mlflow.set_tag("model_family", "logistic_regression")
#         mlflow.log_params(params)

#         # Instanciar y entrenar
#         model = LogisticRegression(**params)
#         model.fit(X_train_bal, y_train_bal)

#         # Validación
#         y_proba = model.predict_proba(X_val)
#         y_pred = model.predict(X_val)

#         # Métricas (multiclase)
#         val_logloss = log_loss(y_val, y_proba)
#         val_acc = accuracy_score(y_val, y_pred)
#         val_f1_macro = f1_score(y_val, y_pred, average="macro")

#         # Log metrics
#         mlflow.log_metric("accuracy", val_acc)
#         mlflow.log_metric("f1_macro", val_f1_macro)

#         # Guardar modelo (anidado)
#         signature = infer_signature(X_val, y_proba)
#         mlflow.sklearn.log_model(model, "model", input_example=X_val[:5], signature=signature)

#     # Optuna minimiza -> devolvemos log_loss
#     return val_logloss

In [ ]:
def objective_logreg(trial: optuna.trial.Trial):
    # --- Hiperparámetros ---
    penalty = trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"])
    C = trial.suggest_float("C", 1e-4, 10.0, log=True)
    l1_ratio = None
    if penalty == "elasticnet":
        l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)

    # --- Param dict ---
    params = {
        "penalty": penalty,
        "C": C,
        "solver": "saga",  # soporta multinomial + regularizaciones
        "multi_class": "multinomial",
        "max_iter": 1000,
        "random_state": 42,
        "n_jobs": -1,
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
    }
    if l1_ratio is not None:
        params["l1_ratio"] = l1_ratio

    # --- Entrenamiento y validación ---
    model = LogisticRegression(**params)
    model.fit(X_train_bal, y_train_bal)

    y_proba = model.predict_proba(X_val_scaled)
    y_pred = model.predict(X_val_scaled)

    val_logloss = log_loss(y_val, y_proba)
    val_f1_macro = f1_score(y_val, y_pred, average="macro")

    # --- Log interno Optuna (solo métricas, sin parámetros duplicados) ---
    mlflow.log_metric("f1_macro", val_f1_macro)
    mlflow.log_metric("log_loss", val_logloss)

    # Retornamos log_loss (Optuna lo minimiza)
    return val_logloss


#### Flujo de búsqueda

In [ ]:
# mlflow.sklearn.autolog(log_models=False)

# sampler = TPESampler(seed=42)
# study = optuna.create_study(direction="minimize", sampler=sampler)

# with mlflow.start_run(run_name="LogisticRegression Hyperparameter Optimization (Optuna)", nested=True):
#     study.optimize(objective_logreg, n_trials=20) 
#     best_params = study.best_params
#     mlflow.log_params(best_params)
#     mlflow.set_tags({
#         "project": "Stress Level Prediction",
#         "optimizer_engine": "optuna",
#         "model_family": "logistic_regression",
#         "feature_set_version": 1
#     })

# final_params = {
#     "solver": "saga",
#     "multi_class": "multinomial",
#     "max_iter": 500,
#     "random_state": 42,
#     "n_jobs": -1
# }

# # combinar best_params
# final_params.update(best_params)

# final_model_lr = LogisticRegression(**final_params)
# final_model_lr.fit(X_train_bal, y_train_bal)

# y_proba = final_model_lr.predict_proba(X_val)
# y_pred = final_model_lr.predict(X_val)

# val_logloss = log_loss(y_val, y_proba)
# val_acc = accuracy_score(y_val, y_pred)
# val_f1_macro = f1_score(y_val, y_pred, average="macro")

# with mlflow.start_run(run_name="LogisticRegression - final", nested=False):
#     mlflow.set_tag("model_family", "logistic_regression")
#     mlflow.log_params(final_params)
#     mlflow.log_metric("val_log_loss", val_logloss)
#     mlflow.log_metric("val_accuracy", val_acc)
#     mlflow.log_metric("val_f1_macro", val_f1_macro)

#     # Guardar el preprocessor (dv)
#     pathlib.Path("preprocessor").mkdir(exist_ok=True)
#     with open("preprocessor/preprocessor.b", "wb") as f_out:
#         pickle.dump(dv, f_out)
#     mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

#     # Log modelo final
#     feature_names_final = features
#     input_example = pd.DataFrame(X_val[:5], columns=feature_names_final)
#     signature = infer_signature(input_example, y_val[:5])


#     mlflow.sklearn.log_model(final_model_lr, "model", input_example=input_example, signature=signature)

# print("Proceso completado.")
# print("val accuracy:", val_acc)


In [ ]:
# Desactivar autolog de modelos (solo params/metrics)
mlflow.sklearn.autolog(log_models=False)

# Definir función objetivo para Optuna
def objective_logreg(trial):
    # Espacio de búsqueda de hiperparámetros
    C = trial.suggest_loguniform("C", 1e-4, 1e2)
    penalty = trial.suggest_categorical("penalty", ["l1", "l2", "elasticnet"])
    l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0) if penalty == "elasticnet" else None
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])

    params = {
        "solver": "saga",
        "multi_class": "multinomial",
        "max_iter": 2000,
        "random_state": 42,
        "n_jobs": -1,
        "C": C,
        "penalty": penalty,
        "l1_ratio": l1_ratio,
        "class_weight": class_weight
    }

    # Cada trial se ejecuta dentro de su propio run en MLflow
    with mlflow.start_run(nested=True):
        mlflow.log_params(params)

        model = LogisticRegression(**params)
        model.fit(X_train_bal, y_train_bal)
        y_pred = model.predict(X_val)

        f1_macro = f1_score(y_val, y_pred, average="macro")
        mlflow.log_metric("val_f1_macro", f1_macro)

    # Retornar el valor negativo (porque se minimiza)
    return -f1_macro


# ============================================================
# CONFIGURACIÓN Y EJECUCIÓN DE OPTUNA
# ============================================================

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="minimize", sampler=sampler)

with mlflow.start_run(run_name="LogisticRegression Hyperparameter Optimization (Optuna)", nested=True):
    study.optimize(objective_logreg, n_trials=100)  # 🔥 más iteraciones
    best_params = study.best_params
    mlflow.log_params(best_params)
    mlflow.set_tags({
        "project": "Stress Level Prediction",
        "optimizer_engine": "optuna",
        "model_family": "logistic_regression",
        "feature_set_version": 1
    })


# ============================================================
# ENTRENAMIENTO FINAL CON LOS MEJORES PARÁMETROS
# ============================================================

final_params = {
    "solver": "saga",
    "multi_class": "multinomial",
    "max_iter": 2000,
    "random_state": 42,
    "n_jobs": -1
}

# Combinar con los mejores encontrados por Optuna
final_params.update(best_params)

final_model_lr = LogisticRegression(**final_params)
final_model_lr.fit(X_train_bal, y_train_bal)

# ============================================================
# EVALUACIÓN FINAL
# ============================================================

y_proba = final_model_lr.predict_proba(X_val)
y_pred = final_model_lr.predict(X_val)

val_logloss = log_loss(y_val, y_proba)
val_acc = accuracy_score(y_val, y_pred)
val_f1_macro = f1_score(y_val, y_pred, average="macro")


# ============================================================
# REGISTRO FINAL EN MLflow
# ============================================================

with mlflow.start_run(run_name="LogisticRegression - final", nested=False):
    mlflow.set_tag("model_family", "logistic_regression")
    mlflow.log_params(final_params)
    mlflow.log_metric("val_log_loss", val_logloss)
    mlflow.log_metric("val_accuracy", val_acc)
    mlflow.log_metric("val_f1_macro", val_f1_macro)

    # Guardar el preprocessor (dv)
    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    # Log del modelo final
    feature_names_final = features
    input_example = pd.DataFrame(X_val[:5], columns=feature_names_final)
    signature = infer_signature(input_example, y_val[:5])

    mlflow.sklearn.log_model(final_model_lr, "model", input_example=input_example, signature=signature)

print("Proceso completado.")
print("Best params:", best_params)
print("val accuracy:", val_acc)
print("val F1 macro:", val_f1_macro)

### Registrar modelo Champion
---

In [ ]:
# model_name = "workspace.default.equipo1-proyecto"

In [ ]:
#runs = mlflow.search_runs(
    #experiment_names=[EXPERIMENT_NAME],
    #order_by=["metrics.val_log_loss ASC"],
    #output_format="list"
#)

# Obtener el mejor run
#if len(runs) > 0:
#    best_run = runs[0]
#    print("🏆 Champion Run encontrado:")
#    print(f"Run ID: {best_run.info.run_id}")
#    print(f"Validation RMSE: {best_run.data.metrics.get('rmse')}")
#    print(f"Params: {best_run.data.params}")
#else:
#    print("⚠️ No se encontraron runs con métrica log_loss.")

In [ ]:
#run_id = best_run.info.run_id

In [ ]:
#result = mlflow.register_model(
    #model_uri=f"runs:/{best_run.info.run_id}/model",
    #name=model_name
#)

In [ ]:
#client = MlflowClient()

#model_version = result.version
#new_alias = "Champion"

#client.set_registered_model_alias(
    #name=model_name,
    #alias=new_alias,
    #version=result.version
#)

#date = datetime.today()

#client.update_model_version(
    #name=model_name,
    #version=model_version,
    #description=f"The model version {model_version} was transitioned to {new_alias} on {date}"
#)

### Random Forest
---

#### Función objetivo

In [ ]:
# Requisitos: importados previamente
# import optuna
# from optuna.samplers import TPESampler
# import mlflow
# import pathlib, pickle
# from mlflow.models.signature import infer_signature
# from sklearn.metrics import log_loss, accuracy_score, f1_score
# from sklearn.ensemble import RandomForestClassifier
# import xgboost as xgb
# mlflow.sklearn.autolog(log_models=False)  # si no lo has hecho aún

# -------------------------
# Objective: Random Forest
# -------------------------
def objective_rf(trial: optuna.trial.Trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "random_state": 42,
        "n_jobs": -1
    }

    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "random_forest")
        mlflow.log_params(params)

        clf = RandomForestClassifier(**params)
        clf.fit(X_train_bal, y_train_bal)

        y_proba = clf.predict_proba(X_val)
        y_pred = clf.predict(X_val)

        val_logloss = log_loss(y_val, y_proba)
        val_acc = accuracy_score(y_val, y_pred)
        val_f1 = f1_score(y_val, y_pred, average="macro")

        mlflow.log_metric("val_log_loss", val_logloss)
        mlflow.log_metric("val_accuracy", val_acc)
        mlflow.log_metric("val_f1_macro", val_f1)

        # Guardar modelo del trial
        signature = infer_signature(X_val, y_val)

        mlflow.sklearn.log_model(clf, "model", input_example=X_val[:5], signature=signature)

    return val_logloss




#### Flujo de búsqueda

In [ ]:
sampler = TPESampler(seed=42)
study_rf = optuna.create_study(direction="minimize", sampler=sampler)

with mlflow.start_run(run_name="RandomForest Hyperparameter Optimization (Optuna)", nested=True):
    study_rf.optimize(objective_rf, n_trials=20)
    best_rf = study_rf.best_params
    mlflow.log_params(best_rf)
    mlflow.set_tags({
        "project": "Stress Level Prediction",
        "optimizer_engine": "optuna",
        "model_family": "random_forest"
    })

# Entrenar RF final con best params y loguear en un run final
final_rf_params = best_rf.copy()
final_rf_params.update({"random_state": 42, "n_jobs": -1})
rf_final = RandomForestClassifier(**final_rf_params)
rf_final.fit(X_train_bal, y_train_bal)
y_val_proba = rf_final.predict_proba(X_val)
y_val_pred = rf_final.predict(X_val)
rf_val_logloss = log_loss(y_val, y_val_proba)
rf_val_acc = accuracy_score(y_val, y_val_pred)
rf_val_f1 = f1_score(y_val, y_val_pred, average="macro")

with mlflow.start_run(run_name="RandomForest - final", nested=False):
    mlflow.log_params(final_rf_params)
    mlflow.log_metric("val_log_loss", rf_val_logloss)
    mlflow.log_metric("val_accuracy", rf_val_acc)
    mlflow.log_metric("val_f1_macro", rf_val_f1)
    # log preprocessor artifact (if not already logged)
    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")
    # log model
    feature_names_final = features
    input_example = pd.DataFrame(X_val[:5], columns=feature_names_final)
    signature = infer_signature(input_example, y_val[:5])
    mlflow.sklearn.log_model(rf_final, "model", input_example=input_example, signature=signature)

print("Random Forest terminado. val_logloss:", rf_val_logloss, "acc:", rf_val_acc, "f1:", rf_val_f1)


### XGBoost
---

#### Función objetivo

In [ ]:

# -------------------------
# Objective: XGBoost
# -------------------------
def objective_xgb(trial: optuna.trial.Trial):
    # xgboost params (sklearn API)

    params = {
        "max_depth": trial.suggest_int("max_depth", 4, 100),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "learning_rate": trial.suggest_float("learning_rate", math.exp(-3), 1.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha",   math.exp(-5), math.exp(-1), log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", math.exp(-6), math.exp(-1), log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", math.exp(-1), math.exp(3), log=True),
        "objective": "reg:squarederror",  
        "seed": 42,                      
    }
    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "xgboost")
        # log params but avoid logging num_class/use_label_encoder maybe
        mlflow.log_params(params)

        clf = xgb.XGBClassifier(**params)
        clf.fit(X_train_bal, y_train_bal, eval_set=[(X_val, y_val)], verbose=False)

        y_proba = clf.predict_proba(X_val)
        y_pred = clf.predict(X_val)

        val_logloss = log_loss(y_val, y_proba)
        val_acc = accuracy_score(y_val, y_pred)
        val_f1 = f1_score(y_val, y_pred, average="macro")

        mlflow.log_metric("val_log_loss", val_logloss)
        mlflow.log_metric("val_accuracy", val_acc)
        mlflow.log_metric("val_f1_macro", val_f1)


        signature = infer_signature(X_test, y_test[:5])

        mlflow.xgboost.log_model(clf, artifact_path="model", input_example=X_test[:5], signature=signature)

    return val_logloss

#### Flujo de búsqueda

In [ ]:
# XGBoost study
study_xgb = optuna.create_study(direction="minimize", sampler=sampler)

with mlflow.start_run(run_name="XGBoost Hyperparameter Optimization (Optuna)", nested=True):
    study_xgb.optimize(objective_xgb, n_trials=20)
    best_xgb = study_xgb.best_params
    mlflow.log_params(best_xgb)
    mlflow.set_tags({
        "project": "Stress Level Prediction",
        "optimizer_engine": "optuna",
        "model_family": "xgboost"
    })

# Entrenar XGB final con best params y loguear en un run final
final_xgb_params = best_xgb.copy()
final_xgb_params.update({
    "objective": "multi:softprob",
    "use_label_encoder": False,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
    "num_class": len(np.unique(y_train_bal))
})
xgb_final = xgb.XGBClassifier(**final_xgb_params)
xgb_final.fit(X_train_bal, y_train_bal)
y_test_proba = xgb_final.predict_proba(X_test)
y_test_pred = xgb_final.predict(X_test)
xgb_test_logloss = log_loss(y_test, y_test_proba)
xgb_test_acc = accuracy_score(y_test, y_test_pred)
xgb_test_f1 = f1_score(y_test, y_test_pred, average="macro")

with mlflow.start_run(run_name="XGBoost - final", nested=False):
    mlflow.log_params(final_xgb_params)
    mlflow.log_metric("test_log_loss", xgb_test_logloss)
    mlflow.log_metric("test_accuracy", xgb_test_acc)
    mlflow.log_metric("test_f1_macro", xgb_test_f1)
    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")
    feature_names_final = features
    input_example = pd.DataFrame(X_test[:5], columns=feature_names_final)
    signature = infer_signature(input_example, y_test[:5])

    # Prefer mlflow.xgboost.log_model, fallback to sklearn flavor
    mlflow.xgboost.log_model(xgb_final, artifact_path="model", input_example=input_example, signature=signature)
    
print("XGBoost terminado. test_logloss:", xgb_test_logloss, "acc:", xgb_test_acc, "f1:", xgb_test_f1)


### Registrar modelo Challenger
---

In [ ]:
#runs = mlflow.search_runs(
    #experiment_names=[EXPERIMENT_NAME],
    #order_by=["metrics.val_log_loss ASC"],
    #output_format="list"
#)

# Obtener el mejor run
#if len(runs) > 0:
#    best_run = runs[0]
#    print("🏆 Champion Run encontrado:")
#    print(f"Run ID: {best_run.info.run_id}")
#    print(f"Validation RMSE: {best_run.data.metrics.get('rmse')}")
#    print(f"Params: {best_run.data.params}")
#else:
#    print("⚠️ No se encontraron runs con métrica log_loss.")

In [ ]:
#run_id = best_run.info.run_id

In [ ]:
#result = mlflow.register_model(
    #model_uri=f"runs:/{best_run.info.run_id}/model",
    #name=model_name
#)

In [ ]:
#client = MlflowClient()

#model_version = result.version
#new_alias = "Challenger"

#client.set_registered_model_alias(
    #name=model_name,
    #alias=new_alias,
    #version=result.version
#)

#date = datetime.today()

#client.update_model_version(
    #name=model_name,
    #version=model_version,
    #description=f"The model version {model_version} was transitioned to {new_alias} on {date}"
#)